# Jury Results Verification — SF1

Reproduces the SF1 jury-vote scoring end-to-end so the official scoring can be independently verified.

Pipeline:
1. **Section 1 — Setup**: imports, file paths, pots, DDI mapping, ISO codes, CSV loading.
2. **Section 2 — Build the rank matrix**: pivot the long-format ESC CSV into a (recipient × voter × juror) matrix.
3. **Section 3 — Score and rank per voting country**: convert ranks to exponential scores, sum per jury, apply within-jury tie-breakers, award 12-10-8-…-1.
4. **Section 4 — Pot substitution** for invalid juries.
5. **Section 5 — Final classification** with PDF §1.3.1 tie-break chain.

Helpers (`rank_to_exp_score`, `rank_to_points`, `rank_jury_with_tiebreakers`, `apply_pot_substitutes`, `rank_final_classification`) live in `jury_helpers.py`.

## Section 1 — Setup

In [ ]:
import os
import math
import numpy as np
import pandas as pd

from jury_helpers import (
    DICT_ISO, ISO_TO_COUNTRY,
    rank_to_exp_score, rank_to_points,
    rank_jury_with_tiebreakers,
    rank_final_classification,
    apply_pot_substitutes,
)


### File paths

`DOB_CSV` is optional. If left at `None` the youngest-juror tie-breaker is skipped, and if a tie ever needs it the notebook halts with a clear warning. Format expected: `EBU_ID;DoB`.

In [ ]:
SOURCE_CSV_DIRECTORY = "./Input"
SCRIPT_XLSX_DIRECTORY = "./Output"

INPUT_CSV = os.path.join(SOURCE_CSV_DIRECTORY, "ESC2025-jurorresults_SF1-Cleaned.csv")
DOB_CSV   = None     # set to a path like os.path.join(SOURCE_CSV_DIRECTORY, "jurors_dob.csv") if available


### Pot membership and show running order

In [ ]:
DICT_POTS_SF1 = {
    "Pot01": ["Croatia", "Finland", "Montenegro", "Serbia", "Sweden"],
    "Pot02": ["Belgium", "Georgia", "Israel", "Moldova", "Poland"],
    "Pot03": ["Estonia", "Greece", "Lithuania", "Portugal", "San Marino"],
}

DICT_POTS_PREQUALIFIED = {
    "Pot00": ["Austria", "France", "Germany", "Italy", "United Kingdom"],
}

# RoW excluded on purpose. PDF page 25 says it is not used in pot substitution.
LIST_POT_00_PREQUALIFIED = ["Austria", "France", "Germany", "Italy", "United Kingdom"]

DICT_DDI_SF1 = {
    "DDI01": "Iceland",     "DDI02": "Poland",      "DDI03": "Slovenia",
    "DDI04": "Estonia",     "DDI05": "Ukraine",     "DDI06": "Sweden",
    "DDI07": "Portugal",    "DDI08": "Norway",      "DDI09": "Belgium",
    "DDI10": "Azerbaijan",  "DDI11": "San Marino",  "DDI12": "Albania",
    "DDI13": "Netherlands", "DDI14": "Croatia",     "DDI15": "Cyprus",
}


### ISO codes

The loader uses `DICT_ISO` (full name → ISO) to map back from the ISO codes in the CSV to full country names, which the rest of the notebook (and the audience notebook) work with.

In [ ]:
# Already imported from jury_helpers; kept here as a sanity check that the keys cover what we need.
print(f"DICT_ISO entries: {len(DICT_ISO)}")
print(f"ISO_TO_COUNTRY entries: {len(ISO_TO_COUNTRY)}")


### Load the jury CSV and (optionally) the DOB CSV

In [ ]:
# ESC2025 jury CSV: long format with columns EBU_ID, ownICO, function, name, votedICO, rank.
raw_df = pd.read_csv(INPUT_CSV, sep=";")
print(f"Loaded {len(raw_df)} rows from {os.path.basename(INPUT_CSV)}")

# Sanity: every ISO code appearing in the CSV must be known.
unknown_voting = sorted(set(raw_df["ownICO"]) - set(ISO_TO_COUNTRY))
unknown_voted  = sorted(set(raw_df["votedICO"]) - set(ISO_TO_COUNTRY))
if unknown_voting or unknown_voted:
    raise ValueError(f"Unknown ISO codes — voting: {unknown_voting}, voted: {unknown_voted}")

# Build the long_df shape the rest of the notebook expects.
long_df = pd.DataFrame({
    "EBU_ID":               raw_df["EBU_ID"],
    "Voting_country":       raw_df["ownICO"].map(ISO_TO_COUNTRY),
    "Juror":                raw_df["function"],
    "Juror_name":           raw_df["name"],
    "Participating_country":raw_df["votedICO"].map(ISO_TO_COUNTRY),
    "Rank":                 raw_df["rank"],
})

print(f"Voting countries: {long_df['Voting_country'].nunique()} | "
      f"Participating: {long_df['Participating_country'].nunique()} | "
      f"Jurors total: {long_df['EBU_ID'].nunique()}")
long_df.head()


In [ ]:
# Optional DOB loader. If absent, juror_dobs stays None; halt if a tie ever needs it.
if DOB_CSV is None:
    juror_dobs_by_id = None
    print("No DOB CSV provided — youngest-juror tie-break disabled.")
else:
    _dob_raw = pd.read_csv(DOB_CSV, sep=";")
    if not {"EBU_ID", "DoB"}.issubset(_dob_raw.columns):
        raise ValueError(f"DOB CSV must have columns EBU_ID;DoB. Got: {list(_dob_raw.columns)}")
    juror_dobs_by_id = _dob_raw.set_index("EBU_ID")["DoB"].to_dict()
    print(f"Loaded DoB for {len(juror_dobs_by_id)} jurors.")


## Section 2 — Build the rank matrix

Pivot the long format into a `(recipient × voter × juror)` matrix. Self-votes (rank=0) survive in the matrix as `0` — the exponential-score helper maps `0` to `0` and they drop out of the per-jury sum.

In [ ]:
participating_order = long_df["Participating_country"].drop_duplicates().tolist()
voting_order        = long_df["Voting_country"].drop_duplicates().tolist()
juror_order         = sorted(long_df["Juror"].unique())   # juror1..juror5

ranks_df = (
    long_df
    .pivot(index="Participating_country",
           columns=["Voting_country", "Juror"],
           values="Rank")
    .reindex(index=participating_order,
             columns=pd.MultiIndex.from_product(
                 [voting_order, juror_order],
                 names=["Voting_country", "Juror"]))
)
print(f"ranks_df shape: {ranks_df.shape}  "
      f"(rows = {len(participating_order)} recipients, "
      f"cols = {len(voting_order)} voters x {len(juror_order)} jurors)")
ranks_df.head()


In [ ]:
# Lookup table from (Voting_country, Juror) -> EBU_ID. Needed downstream so we can
# look up a juror's DoB from the DOB CSV (which is keyed on EBU_ID, not on juror1/2/...).
juror_id_lookup = (
    long_df
    .drop_duplicates(["Voting_country", "Juror"])
    .set_index(["Voting_country", "Juror"])["EBU_ID"]
)

# Show running order = DDI number, taken straight from DICT_DDI_SF1.
# DDI01 = first to perform, used as the final tie-break in PDF §1.3.1 (earlier wins).
show_order = {country: int(ddi.replace("DDI", "")) for ddi, country in DICT_DDI_SF1.items()}
print(f"Show order built for {len(show_order)} participants.")


## Section 3 — Score and rank per voting country

### 3.1 Convert ranks to exponential scores

In [ ]:
exp_scores_df = ranks_df.map(rank_to_exp_score)
exp_scores_df.head()


### 3.2 Sum per jury, rank with within-jury tie-breakers, award points

Within-jury tie-breaker chain (PDF §1.3.1, applied when two contenders share the same exponential sum within a single voting country):

1. Majority of better individual rankings among that jury's jurors.
2. Vote of the youngest juror — **requires `DOB_CSV` to be provided**.
3. Show of hands — interactive prompt for the winning country.

If step 2 is needed and no DOB CSV was loaded, the notebook halts with a banner.

In [ ]:
# Sum exponential scores per voting country (drop the per-juror level).
jury_sums = exp_scores_df.T.groupby(level="Voting_country", sort=False).sum().T

# Mask self-votes so a country cannot rank itself.
jury_sums_for_ranking = jury_sums.copy()
for vc in jury_sums_for_ranking.columns:
    if vc in jury_sums_for_ranking.index:
        jury_sums_for_ranking.loc[vc, vc] = pd.NA


In [ ]:
# DOB-required halt logic. We pass a sentinel dict to the helper; any attempt to read a
# juror DOB raises DOBNeededError, which we trap and convert into the halt banner.

class DOBNeededError(Exception):
    def __init__(self, voting_country, juror_key):
        self.voting_country = voting_country
        self.juror_key = juror_key
        super().__init__(f"DOB needed for {voting_country!r}, juror {juror_key!r}")


class _DOBSentinel(dict):
    """Stand-in for a DoB lookup dict. Reading any key raises DOBNeededError."""
    def __init__(self, voting_country):
        super().__init__()
        self._vc = voting_country
    def __getitem__(self, key):
        raise DOBNeededError(self._vc, key)
    def get(self, key, default=None):
        raise DOBNeededError(self._vc, key)


def _build_dobs_for_jury(vc):
    """Return a real DoB dict for voting country vc, or a sentinel if no DOB CSV was loaded."""
    if juror_dobs_by_id is None:
        return _DOBSentinel(vc)
    out = {}
    for juror, ebu_id in juror_id_lookup.loc[vc].items():
        out[juror] = juror_dobs_by_id.get(ebu_id)
    return out


def _halt_age_required(vc, juror_key):
    banner = "=" * 64
    print()
    print(banner)
    print("HALT — JUROR DATE OF BIRTH REQUIRED")
    print(banner)
    print(f"Within-jury tie at {vc!r} could not be resolved by the majority rule")
    print(f"(PDF §1.3.1 box 1). The next step is the youngest-juror tie-break")
    print(f"(PDF §1.3.1 box 2), which needs each juror's date of birth.")
    print()
    print(f"  voting country : {vc}")
    print(f"  juror needed   : {juror_key}")
    print()
    print("Provide a DOB CSV via the DOB_CSV variable in Section 1 and re-run.")
    print("Expected schema: EBU_ID;DoB")
    print(banner)
    raise SystemExit(1)


In [ ]:
jury_ranks = pd.DataFrame(index=jury_sums.index, columns=jury_sums.columns, dtype="Int64")

for vc in jury_sums.columns:
    sums = jury_sums_for_ranking[vc]
    ranks_in_jury = ranks_df.xs(vc, axis=1, level="Voting_country")
    dobs_for_jury = _build_dobs_for_jury(vc)
    try:
        rank_series = rank_jury_with_tiebreakers(vc, sums, ranks_in_jury, dobs_for_jury)
    except DOBNeededError as e:
        _halt_age_required(e.voting_country, e.juror_key)
    for c, r in rank_series.items():
        jury_ranks.loc[c, vc] = r

jury_points = jury_ranks.map(rank_to_points).astype(int)
print(f"jury_points shape: {jury_points.shape}")
jury_points


## Section 4 — Pot substitution for invalid juries

A national jury is invalid if it has fewer than 3 valid jurors (per EBU §1.3) or if it has been disqualified by the EBU. Add country names to `manual_disqualified` to force pot substitution for testing.

In [ ]:
manual_disqualified = []   # add country names here to force pot substitution

# Build pot_assignments dict from DICT_POTS_SF1: {country: pot_number}
pot_assignments = {}
for pot_label, members in DICT_POTS_SF1.items():
    pot_num = int(pot_label.replace("Pot", ""))
    for c in members:
        pot_assignments[c] = pot_num
for c in LIST_POT_00_PREQUALIFIED:
    pot_assignments[c] = 0

jury_points, substituted = apply_pot_substitutes(
    jury_points, ranks_df, pot_assignments, show_order,
    long_df, manual_disqualified=manual_disqualified,
)
if substituted:
    print(f"Pot substitution applied for: {substituted}")
else:
    print("No pot substitution needed.")


## Section 5 — Final classification

Final-tie chain (PDF §1.3.1, when two countries have the same total points):
1. Highest number of juries that gave it any points.
2. Highest number of 12-point scores; then 10s, 8s, 7s, 6s, 5s, 4s, 3s, 2s, 1s.
3. Earlier in the show running order wins.

In [ ]:
final_classification = rank_final_classification(jury_points, show_order)
final_classification


## Run summary

In [ ]:
print("=" * 60)
print(f"Run complete — input: {os.path.basename(INPUT_CSV)}")
print(f"DOB tie-break enabled: {juror_dobs_by_id is not None}")
print(f"Pot substitutions applied: {substituted if substituted else 'none'}")
print("=" * 60)
